In [232]:
import tkinter as tk
from tkinter import ttk 
from ttkthemes import ThemedTk
import math
import matplotlib.pyplot as plt
from numpy import arange, asarray, exp, array, float32
from numpy.random import normal
import numpy as np

from pipython import GCSDevice, pitools
from pipython.pidevice.gcsmessages import GCSMessages
from pipython.pidevice.interfaces.piserial import PISerial
from pipython.pidevice.gcscommands import GCSCommands
from sys import platform
import time

from matplotlib.backend_bases import key_press_handler
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure

In [233]:
def gaussian(x, sigma, mu):
    x = asarray(x)
    #y= (1/(sigma * math.sqrt(2 * math.pi )) * math.exp(-1/2 * (x-mu)**2 / sigma**2))
    return asarray(1/(sigma * math.sqrt(2 * math.pi )) * exp(-1/2 * (x-mu)**2 / sigma**2))

def define_x_grid(min, max, resolution):

    x = arange(min, max+resolution, resolution)

    return x

def create_random_gaussian(with_noise = False):
    random_x = np.random.Generator.random(-10, 10, size=None)
    y = gaussian(random_x, 1, 0)
    if with_noise == True:
        y += float(normal(scale=0.1, size=None))
    return random_x, y

def append_point(x, y, x_new, y_new):
    x.append(x_new)
    y.append(y_new)
    

In [234]:
class Spectral_Data():
    def __init__(self):
        self.data = np.empty((2,0), dtype=float32)
        self.rng = np.random.default_rng()

    def add_point(self, x, y):
        point = asarray([[x],[y]], dtype=float32)
        self.data = np.append(self.data, point, axis=1)
        #self.update_plot()

    def print_data(self):
        print(self.data)

    def clear_data(self):
        self.data = np.empty((2,0), dtype=float32)

    def add_random_point(self):
        new_x = self.rng.uniform(low=-5, high=5)
        new_y = gaussian(new_x, 1.44, 0)
        self.add_point(new_x, new_y)

    def add_noise(self):
        self.data[1,:] += self.rng.normal(scale=0.01, size=len(self.data[1,:]))
        #self.plot_points.set_offsets(self.data.T)
        #self.fig.canvas.draw_idle()


class Spectral_plot():
    def __init__(self, window):
        self.fig = Figure(dpi=75)
        self.ax = self.fig.add_subplot()
        self.plot_points = self.ax.scatter([], [])
        self.window = window
        self.window.columnconfigure(0, weight=1)
        self.window.rowconfigure(0, weight=1)

        self.canvas = FigureCanvasTkAgg(self.fig, master=self.window)
        self.canvas.draw()
        self.canvas.get_tk_widget().grid(row=0, column=0, sticky="nsew")       

    #def plot_spectrum(self):
    #    plt.scatter(spectrum.data[0,:], spectrum.data[1,:])

    def update_plot(self, data):
        self.plot_points.set_offsets(data.T)
        self.ax.update_datalim(data.T, updatex=True, updatey=True)
        self.ax.autoscale_view()
        self.fig.canvas.draw_idle()

    #def clear_plot(self):
    #    self.update_plot(np.empty((2,0), dtype=float32))

class Control_stage():
    def __init__(self):
       # if platform == "linux" or platform == "linux2":
       #     self.port = '/dev/ttyS0',
       # elif platform == "win32":
       #     self.port='COM1'

        self.pidevice = GCSDevice()
        self.device = tk.StringVar(value=None)
        self.devices_list = ["test"]

    def get_devices(self):
        self.devices_list = list(self.pidevice.EnumerateTCPIPDevices(mask='C-884.4DB'))
        if len(self.devices_list) != 0:
            return self.devices_list
        else:
            return []
    
    def connect_device(self, device):
        self.pidevice.ConnectTCPIPByDescription(device)

    def print_identity(self):
        self.pidevice.qIDN()

    def move_stage_to_z(self, z_position):
        pitools.moveandwait(self.pidevice, 'Axis_1', float32(z_position))

class Control_stage_fake():
    def __init__(self):
        self.pidevice = "test"
        self.pos = 0
        self.device = tk.StringVar(value=None)
        self.devices_list = ["test"]

    def get_devices(self):
        self.devices_list = ["Device 1", "Device 2", "Device 3", "Device 4"]
        if len(self.devices_list) != 0:
            return self.devices_list
        else:
            return []
    
    def connect_device(self, device):
        self.pidevice = device

    def print_identity(self):
        return self.pidevice

    def move_stage_to_z(self, z_position):
        self.pos = z_position
        time.sleep(1)

class Gui():
    def __init__(self, window, title, geo, stage):
        self.window = window
        self.window.title(title)
        self.window.geometry(geo)
        self.window.config(background="lightgrey")
        self.window.columnconfigure(0,weight=0)
        self.window.columnconfigure(1,weight=1)
        #self.window.columnconfigure(2,weight=1)
        self.window.rowconfigure(0,weight=1)
        #self.window.rowconfigure(1,weight=0)
        #self.window.rowconfigure(2,weight=1)

        self.data = Spectral_Data()
        self.stage = stage

        self.frame_plot = ttk.Frame(self.window)
        self.frame_plot.columnconfigure(0,weight=1)
        self.frame_plot.rowconfigure(0,weight=1)
        self.frame_plot.grid(row=0, column=1,padx=5, pady=5, sticky="nsew")

        self.plot = Spectral_plot(self.frame_plot)

        self.frame_controls = ttk.Frame(self.window)
        self.frame_controls.grid(row=0, column=0,padx=5, pady=5, sticky="nsew")

########################################################## Frame 1 ##########################################################
        self.frame1 = ttk.Frame(self.frame_controls)
        self.frame1.grid(row    = 0, 
                         column = 0,
                         padx   = 5, 
                         pady   = 5, 
                         sticky = "nsew")

        self.greeting = ttk.Label(self.frame1, text="wassup")
        self.greeting.grid(row          = 3, 
                           column       = 0, 
                           columnspan   = 2, 
                           padx         = 5, 
                           pady         = 5,
                           sticky="ew")

        self.off_button = ttk.Button(self.frame1, text="OFF", command=self.window.destroy)
        self.off_button.grid(row    = 0, 
                             column = 0,
                             padx   = 5, 
                             pady   = 5, 
                             sticky = "ew")

        self.add_point_button =ttk.Button(self.frame1, text="add", command=self.add_random_point)
        self.add_point_button.grid(row      = 0, 
                                   column   = 1,
                                   padx     = 5, 
                                   pady     = 5, 
                                   sticky   = "ew")

        self.clear_button = ttk.Button(self.frame1, text="clear", command=self.clear_plot)
        self.clear_button.grid(row      = 1, 
                               column   = 0,
                               padx     = 5, 
                               pady     = 5, 
                               sticky   = "ew")

        self.noise_button = ttk.Button(self.frame1, text="noise", command=self.add_noise)
        self.noise_button.grid(row      = 1, 
                               column   = 1,
                               padx     = 5, 
                               pady     = 5, 
                               sticky   = "ew")

########################################################## Frame 2 ##########################################################
        self.frame2 = ttk.Frame(self.frame_controls)
        self.frame2.grid(row    = 1, 
                         column = 0,
                         padx   = 5, 
                         pady   = 5, 
                         sticky = "nsew")
        

        self.z_stage_label = ttk.Label(self.frame2, text="Position Stage:")
        self.z_stage_label.grid(row     = 0, 
                                column  = 0,
                                padx    = 5, 
                                pady    = 5)

        self.enter_z_stage = ttk.Entry(self.frame2)
        self.enter_z_stage.grid(row     = 0, 
                                column  = 1,
                                padx    = 5, 
                                pady    = 5, 
                                sticky  = "ew")
        
        self.get_devices = ttk.Button(self.frame2, text="Get Devices", command=self.update_device_list)
        self.get_devices.grid(row      = 1, 
                               column   = 0,
                               padx     = 5, 
                               pady     = 5, 
                               sticky   = "ew")


        self.z_stage_select_controler = ttk.Combobox(self.frame2, state="readonly", textvariable=self.stage.device, values=self.stage.devices_list)
        self.z_stage_select_controler.grid( row     = 1, 
                                            column  = 1,
                                            padx    = 5, 
                                            pady    = 5, 
                                            sticky  = "ew")

    def update_device_list(self):
        self.stage.get_devices()
        self.z_stage_select_controler["values"] = self.stage.devices_list

    def add_random_point(self):
        self.data.add_random_point()
        self.plot.update_plot(self.data.data)

    def add_noise(self):
        self.data.add_noise()
        self.plot.update_plot(self.data.data)

    def clear_plot(self):
        self.data.clear_data()
        self.plot.update_plot(self.data.data)



In [235]:

window = ThemedTk(theme='yaru')
stage = Control_stage_fake()
Gui_main = Gui(window, "Data Aquisition", "700x400", stage)


In [236]:
window.mainloop()